# Lab | Training Deep Networks

## Overview
In this lab, we train a four-layer fully-connected neural network on the Fashion-MNIST dataset. We use a canonical PyTorch training loop with batch normalization, dropout, an Adam optimizer, and a cosine annealing learning rate scheduler.

## Setup and Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
print(f"Using device: {device}")

Using device: cpu


## Data Loading
Loading Fashion-MNIST and creating train/validation DataLoaders.

In [2]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=tf)
val_set   = datasets.FashionMNIST(root="./data", train=False, download=True, transform=tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)

## Task 1 — Train the Network
### Model Architecture
Defining a four-layer fully-connected ReLU network with batch normalization and dropout.

In [3]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)

print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (3): ReLU()
  (4): Dropout(p=0.3, inplace=False)
  (5): Linear(in_features=256, out_features=128, bias=True)
  (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (7): ReLU()
  (8): Dropout(p=0.3, inplace=False)
  (9): Linear(in_features=128, out_features=64, bias=True)
  (10): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (11): ReLU()
  (12): Linear(in_features=64, out_features=10, bias=True)
)


### Training Configuration

In [4]:
epochs = 15
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

### Training Loop

In [5]:
def calculate_metrics(loader, model, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss / total, correct / total

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
    scheduler.step()
    
    train_loss, train_acc = calculate_metrics(train_loader, model, criterion)
    val_loss, val_acc = calculate_metrics(val_loader, model, criterion)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1}/{epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

Epoch 1/15 - Train Loss: 0.3696, Train Acc: 0.8651, Val Loss: 0.4139, Val Acc: 0.8510
Epoch 2/15 - Train Loss: 0.3169, Train Acc: 0.8857, Val Loss: 0.3677, Val Acc: 0.8682
Epoch 3/15 - Train Loss: 0.3176, Train Acc: 0.8819, Val Loss: 0.3738, Val Acc: 0.8635
Epoch 4/15 - Train Loss: 0.2809, Train Acc: 0.8974, Val Loss: 0.3426, Val Acc: 0.8753
Epoch 5/15 - Train Loss: 0.2611, Train Acc: 0.9035, Val Loss: 0.3331, Val Acc: 0.8779
Epoch 6/15 - Train Loss: 0.2437, Train Acc: 0.9096, Val Loss: 0.3232, Val Acc: 0.8835
Epoch 7/15 - Train Loss: 0.2402, Train Acc: 0.9116, Val Loss: 0.3237, Val Acc: 0.8836
Epoch 8/15 - Train Loss: 0.2287, Train Acc: 0.9160, Val Loss: 0.3225, Val Acc: 0.8854
Epoch 9/15 - Train Loss: 0.2102, Train Acc: 0.9226, Val Loss: 0.3087, Val Acc: 0.8873
Epoch 10/15 - Train Loss: 0.1991, Train Acc: 0.9271, Val Loss: 0.3046, Val Acc: 0.8895
Epoch 11/15 - Train Loss: 0.1883, Train Acc: 0.9312, Val Loss: 0.2910, Val Acc: 0.8951
Epoch 12/15 - Train Loss: 0.1826, Train Acc: 0.9335,

### Results Visualization

In [6]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Val Accuracy')
plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

best_val_acc = max(val_accs)
best_epoch = val_accs.index(best_val_acc) + 1
print(f"Best Validation Accuracy: {best_val_acc:.4f} at Epoch {best_epoch}")

Best Validation Accuracy: 0.8972 at Epoch 15


### Interpretation
The training and validation loss curves decrease steadily and stay close together, indicating strong generalization with minimal overfitting. The model reached its best validation accuracy of 89.72% at epoch 15, showing that performance was still slightly improving towards the end of the 15-epoch run. This result aligns perfectly with the expected 89-90% range for this architecture.